# Generate (n, k, m, P) Training Data with LP-computed m-Heights

Generates 10,000 random samples and computes their exact m-heights using the LP-based algorithm.

- **n** = 9 (fixed)
- **k** ∈ {4, 5, 6}
- **m** ∈ {2, ..., n−k}
- **P** is a k × (n−k) matrix of integers drawn uniformly from [−100, 100]

In [10]:
import numpy as np
from scipy.optimize import linprog
from itertools import combinations, product
from multiprocessing import Pool, cpu_count
import pickle
import time
import sys

## 1. Generate 10,000 random (n, k, m, P) samples

We draw uniformly across the 9 valid (k, m) combinations, with P entries as random integers in [−100, 100].

In [ ]:
N_SAMPLES = 10_000
N = 9
rng = np.random.default_rng(seed=2026)

# All valid (k, m) combinations
configs = []
for k in [4, 5, 6]:
    for m in range(2, N - k + 1):  # m in {2, ..., n-k}
        configs.append((k, m))

print(f"Valid (k, m) combinations ({len(configs)} total):")
for k, m in configs:
    print(f"  k={k}, m={m}  -> P shape: ({k}, {N-k})")

def min_distance(P_mat, k, n):
    """Compute minimum distance d of the code [I_k | P] via parity check matrix.
    d = smallest number of linearly dependent columns of H = [-P^T | I_{n-k}]."""
    r = n - k
    H = np.hstack([-P_mat.T, np.eye(r)])
    for t in range(1, n + 1):
        for cols in combinations(range(n), t):
            sub = H[:, list(cols)]
            if np.linalg.matrix_rank(sub, tol=1e-8) < min(t, r):
                return t
    return n + 1

# Generate samples: uniform across (k, m) groups, ensuring m < d (finite m-height)
samples = []
rejected = 0
for i in range(N_SAMPLES):
    while True:
        k, m = configs[rng.integers(len(configs))]
        P = rng.integers(-100, 101, size=(k, N - k))
        d = min_distance(P, k, N)
        if m < d:
            samples.append([N, k, m, P])
            break
        rejected += 1

# Verify distribution
from collections import Counter
km_counts = Counter((s[1], s[2]) for s in samples)
print(f"\nGenerated {len(samples)} samples ({rejected} rejected for m >= d). Distribution:")
for key in sorted(km_counts.keys()):
    print(f"  k={key[0]}, m={key[1]}: {km_counts[key]} samples")

# Quick sanity check
s = samples[0]
print(f"\nSample 0: n={s[0]}, k={s[1]}, m={s[2]}, P shape={s[3].shape}, "
      f"P range=[{s[3].min()}, {s[3].max()}]")

Valid (k, m) combinations (9 total):
  k=4, m=2  -> P shape: (4, 5)
  k=4, m=3  -> P shape: (4, 5)
  k=4, m=4  -> P shape: (4, 5)
  k=4, m=5  -> P shape: (4, 5)
  k=5, m=2  -> P shape: (5, 4)
  k=5, m=3  -> P shape: (5, 4)
  k=5, m=4  -> P shape: (5, 4)
  k=6, m=2  -> P shape: (6, 3)
  k=6, m=3  -> P shape: (6, 3)


## 2. LP-based m-Height computation

Exact LP algorithm from the project spec (Theorem 1). Each sample is solved independently, so we parallelize across CPU cores.

In [12]:

def compute_m_height(n, k, m, P):
    """
    Compute m-height using the simplified LP algorithm from:
    Roth et al., "On the Height Profile of Analog Error-Correcting Codes"
    (Theorem 1 + Figure 1).

    For each m-subset S of [n) (the "free" positions that can be large):
      - Constrain entries at positions S_bar to [-1, 1]:  -1 <= u·g_j <= 1 for j in S_bar
      - For each i in S, maximize u·g_i

    Total LPs: m * C(n, m).
    """
    G = np.hstack([np.eye(k), P])
    g_cols = G.T  # g_cols[j] = j-th column of G (length k)

    h = 0.0
    indices = list(range(n))

    # Enumerate all m-subsets S of [n) — these are the m "free" positions
    for S in combinations(indices, m):
        S_set = set(S)
        S_bar = [j for j in indices if j not in S_set]  # complement, size n-m

        # Constraints on S_bar: -1 <= u · g_j <= 1 for all j in S_bar
        G_Sbar = np.array([g_cols[j] for j in S_bar])  # (n-m, k)
        A_ub = np.vstack([G_Sbar, -G_Sbar])             # (2(n-m), k)
        b_ub = np.ones(2 * (n - m))

        # For each i in S (the free positions), maximize u · g_i
        for i in S:
            c_obj = -g_cols[i]  # negate for minimization
            result = linprog(c_obj, A_ub=A_ub, b_ub=b_ub,
                             bounds=[(None, None)] * k,
                             method='highs',
                             options={'presolve': True,
                                      'dual_feasibility_tolerance': 1e-8,
                                      'primal_feasibility_tolerance': 1e-8})
            if result.success:
                val = -result.fun
                if val > h:
                    h = val
            elif result.status == 3:  # unbounded
                return float('inf')

    return h


def _worker(args):
    """Worker function for multiprocessing."""
    idx, n, k, m, P = args
    height = compute_m_height(n, k, m, P)
    return idx, height

print(f"LP solver ready (new algorithm). Available CPUs: {cpu_count()}")

# Quick comparison of LP counts: old vs new
from math import comb
print("\nLP count comparison (n=9):")
print(f"{'k':>3} {'m':>3} | {'Old algorithm':>14} | {'New algorithm':>14} | {'Speedup':>8}")
print("-" * 55)
for k_val in [4, 5, 6]:
    for m_val in range(2, 9 - k_val + 1):
        old = 9 * 8 * comb(7, m_val - 1) * (2 ** m_val)
        new = m_val * comb(9, m_val)
        print(f"{k_val:>3} {m_val:>3} | {old:>14,} | {new:>14,} | {old/new:>7.1f}x")


LP solver ready (new algorithm). Available CPUs: 32

LP count comparison (n=9):
  k   m |  Old algorithm |  New algorithm |  Speedup
-------------------------------------------------------
  4   2 |          2,016 |             72 |    28.0x
  4   3 |         12,096 |            252 |    48.0x
  4   4 |         40,320 |            504 |    80.0x
  4   5 |         80,640 |            630 |   128.0x
  5   2 |          2,016 |             72 |    28.0x
  5   3 |         12,096 |            252 |    48.0x
  5   4 |         40,320 |            504 |    80.0x
  6   2 |          2,016 |             72 |    28.0x
  6   3 |         12,096 |            252 |    48.0x


## 3. Compute m-heights in parallel

In [13]:
# Prepare work items
work = [(i, s[0], s[1], s[2], s[3]) for i, s in enumerate(samples)]

n_workers = max(1, cpu_count() - 1)
print(f"Computing m-heights for {len(work)} samples using {n_workers} workers...")
sys.stdout.flush()

m_heights = [None] * len(samples)
start = time.time()
done = 0

with Pool(n_workers) as pool:
    for idx, height in pool.imap_unordered(_worker, work, chunksize=10):
        m_heights[idx] = height
        done += 1
        if done % 200 == 0 or done == len(work):
            elapsed = time.time() - start
            rate = done / elapsed
            eta = (len(work) - done) / rate if rate > 0 else 0
            print(f"  [{done}/{len(work)}] {elapsed:.0f}s elapsed, "
                  f"{rate:.1f} samples/s, ETA {eta:.0f}s")
            sys.stdout.flush()

elapsed = time.time() - start
print(f"\nDone! {len(m_heights)} m-heights computed in {elapsed:.1f}s")
print(f"Sample heights (first 10): {m_heights[:10]}")

Computing m-heights for 10000 samples using 31 workers...
  [200/10000] 2s elapsed, 99.3 samples/s, ETA 99s
  [400/10000] 3s elapsed, 129.4 samples/s, ETA 74s
  [600/10000] 4s elapsed, 139.8 samples/s, ETA 67s
  [800/10000] 5s elapsed, 154.6 samples/s, ETA 60s
  [1000/10000] 6s elapsed, 161.2 samples/s, ETA 56s
  [1200/10000] 7s elapsed, 164.0 samples/s, ETA 54s
  [1400/10000] 8s elapsed, 173.6 samples/s, ETA 50s
  [1600/10000] 9s elapsed, 172.6 samples/s, ETA 49s
  [1800/10000] 10s elapsed, 173.4 samples/s, ETA 47s
  [2000/10000] 11s elapsed, 176.8 samples/s, ETA 45s
  [2200/10000] 12s elapsed, 178.5 samples/s, ETA 44s
  [2400/10000] 13s elapsed, 180.5 samples/s, ETA 42s
  [2600/10000] 14s elapsed, 182.9 samples/s, ETA 40s
  [2800/10000] 15s elapsed, 182.0 samples/s, ETA 40s
  [3000/10000] 16s elapsed, 185.2 samples/s, ETA 38s
  [3200/10000] 17s elapsed, 186.2 samples/s, ETA 37s
  [3400/10000] 18s elapsed, 186.1 samples/s, ETA 35s
  [3600/10000] 19s elapsed, 189.3 samples/s, ETA 34s
 

## 4. Save to pickle files

In [15]:
out_dir = '/workspace/Homework/Project1'

with open(f'{out_dir}/Generated_n_k_m_P', 'wb') as f:
    pickle.dump(samples, f)

with open(f'{out_dir}/Generated_m-heights', 'wb') as f:
    pickle.dump(m_heights, f)

print(f"Saved {len(samples)} samples to Generated_n_k_m_P")
print(f"Saved {len(m_heights)} m-heights to Generated_m-heights")

# Verify by reloading
with open(f'{out_dir}/Generated_n_k_m_P', 'rb') as f:
    check_data = pickle.load(f)
with open(f'{out_dir}/Generated_m-heights', 'rb') as f:
    check_heights = pickle.load(f)
print(f"Verification: {len(check_data)} samples, {len(check_heights)} heights")
print(f"Heights range: [{min(check_heights):.6f}, {max(check_heights):.6f}]")

Saved 10000 samples to Generated_n_k_m_P
Saved 10000 m-heights to Generated_m-heights
Verification: 10000 samples, 10000 heights
Heights range: [30.279583, inf]
